# TARS: reproducible training run

Retrains the TARS classifier with **the same architecture and hyperparameters as the original notebook**, but with a saved, fixed data split so every reported number can be reproduced.

What is different from the original training:
- **Three-way split** (train / validation / test, stratified, fixed seed). Validation chooses the epoch and the threshold; **test is touched once, at the end**, and is what the paper reports.
- The exact file lists, per-epoch history, chosen epoch, threshold and test predictions are all saved.
- Optional **leave-regions-out** mode: train on some regions, test on regions never seen in training (region code is in every file name).

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. About 30–40 min. When it finishes, download `tars_run_outputs.zip` (last cell) and send it back.

In [ ]:
# ---- Settings -------------------------------------------------------------
SPLIT_MODE = "random"          # "random" or "regions"
HOLDOUT_REGIONS = ["SIN"]      # used only when SPLIT_MODE == "regions"
SEED = 42
EPOCHS = 12
BATCH = 16
IMG = 380
VAL_FRAC, TEST_FRAC = 0.15, 0.15
RUN_NAME = f"tars_{SPLIT_MODE}" + ("_" + "-".join(HOLDOUT_REGIONS) if SPLIT_MODE == "regions" else "")
print(RUN_NAME)

In [ ]:
import os, re, json, glob, shutil, platform
import numpy as np, pandas as pd, cv2, tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
tf.keras.utils.set_random_seed(SEED)
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
OUT = f"/content/{RUN_NAME}"; os.makedirs(OUT, exist_ok=True)

## 1. Data: download, clean, split (all decisions saved)

In [ ]:
import kagglehub
root = kagglehub.dataset_download("harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset")
files = sorted(glob.glob(os.path.join(root, "**", "S1SAR_UnBalanced_400by400_Class_*", "**", "*.jpg"), recursive=True))
rows = []
for f in files:
    label = int(re.search(r"Class_([01])", f).group(1))
    region = re.search(r"_([A-Za-z]+)_cls_", os.path.basename(f)).group(1)
    g = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    white = float(np.mean(g > 250)) if g is not None else 1.0
    rows.append(dict(path=f, file=os.path.basename(f), label=label, region=region, white_frac=white))
df = pd.DataFrame(rows).drop_duplicates("file")
# Same cleaning rule as the original notebook: drop Clean-Sea chips that are mostly pure white.
df["excluded"] = (df.label == 0) & (df.white_frac > 0.5)
print(len(df), "chips |", df.excluded.sum(), "excluded as corrupted")
data = df[~df.excluded].copy()

In [ ]:
if SPLIT_MODE == "random":
    trainval, test = train_test_split(data, test_size=TEST_FRAC, stratify=data.label, random_state=SEED)
else:
    test = data[data.region.isin(HOLDOUT_REGIONS)]
    trainval = data[~data.region.isin(HOLDOUT_REGIONS)]
train, val = train_test_split(trainval, test_size=VAL_FRAC / (1 - TEST_FRAC) if SPLIT_MODE == "random" else VAL_FRAC,
                              stratify=trainval.label, random_state=SEED)
data["split"] = "train"
data.loc[data.file.isin(val.file), "split"] = "val"
data.loc[data.file.isin(test.file), "split"] = "test"
manifest = pd.concat([data, df[df.excluded].assign(split="excluded")])
manifest[["file", "label", "region", "white_frac", "split"]].to_csv(f"{OUT}/split_manifest.csv", index=False)
print(data.groupby(["split", "label"]).size().unstack())

## 2. Model and training (identical to the original)

In [ ]:
pre = tf.keras.applications.efficientnet.preprocess_input
aug = ImageDataGenerator(preprocessing_function=pre, rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
                         shear_range=0.2, zoom_range=0.2, horizontal_flip=True, vertical_flip=True, fill_mode="reflect")
plain = ImageDataGenerator(preprocessing_function=pre)
def flow(gen, frame, shuffle):
    return gen.flow_from_dataframe(frame.assign(y=frame.label.astype(str)), x_col="path", y_col="y",
                                   target_size=(IMG, IMG), batch_size=BATCH, class_mode="binary",
                                   shuffle=shuffle, seed=SEED)
train_gen = flow(aug, data[data.split == "train"], True)
val_gen = flow(plain, data[data.split == "val"], False)

n0, n1 = (data[data.split == "train"].label == 0).sum(), (data[data.split == "train"].label == 1).sum()
class_weight = {0: (n0 + n1) / (2 * n0), 1: (n0 + n1) / (2 * n1)}
print("class weights", class_weight)

base = applications.EfficientNetB4(weights="imagenet", include_top=False, input_shape=(IMG, IMG, 3))
base.trainable = False
model = models.Sequential([base, layers.GlobalAveragePooling2D(), layers.BatchNormalization(), layers.Dropout(0.4),
                           layers.Dense(512, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.5),
                           layers.Dense(1, activation="sigmoid")])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
ckpt = f"{OUT}/epoch_{{epoch:02d}}.weights.h5"
history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, class_weight=class_weight,
                    callbacks=[tf.keras.callbacks.ModelCheckpoint(ckpt, save_weights_only=True)])
hist = pd.DataFrame(history.history); hist.index += 1; hist.to_csv(f"{OUT}/history.csv", index_label="epoch")
hist

## 3. Choose epoch and threshold on validation only

In [ ]:
best_epoch = int(hist.val_auc.idxmax())
model.load_weights(f"{OUT}/epoch_{best_epoch:02d}.weights.h5")
model.save_weights(f"{OUT}/tars_b4_selected.weights.h5")
val_scores = model.predict(val_gen, verbose=0).ravel(); yv = val_gen.classes
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
f1s = [f1_score(yv, (val_scores > t).astype(int)) for t in grid]
threshold = float(grid[int(np.argmax(f1s))])
print(f"selected epoch {best_epoch} (val AUC {hist.val_auc[best_epoch]:.4f}); threshold {threshold} (val F1 {max(f1s):.4f})")

## 4. Test set: evaluated once

In [ ]:
test_gen = flow(plain, data[data.split == "test"], False)
ts = model.predict(test_gen, verbose=0).ravel(); yt = test_gen.classes; pt = (ts > threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(yt, pt).ravel()
metrics = dict(run=RUN_NAME, split_mode=SPLIT_MODE, holdout_regions=HOLDOUT_REGIONS if SPLIT_MODE == "regions" else None,
               selected_epoch=best_epoch, threshold=threshold, n_test=int(len(yt)),
               test_auc=float(roc_auc_score(yt, ts)) if len(set(yt)) > 1 else None,
               test_accuracy=float((tp + tn) / len(yt)), test_precision=float(precision_score(yt, pt, zero_division=0)),
               test_recall=float(recall_score(yt, pt, zero_division=0)), test_f1=float(f1_score(yt, pt, zero_division=0)),
               confusion=dict(TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp)),
               tf_version=tf.__version__, python=platform.python_version())
json.dump(metrics, open(f"{OUT}/metrics.json", "w"), indent=2)
pd.DataFrame({"file": [os.path.basename(p) for p in test_gen.filepaths], "label": yt, "score": ts}).to_csv(f"{OUT}/test_predictions.csv", index=False)
print(json.dumps(metrics, indent=2))

In [ ]:
# Per-region test results (most useful in "random" mode)
pred = pd.read_csv(f"{OUT}/test_predictions.csv").merge(data[["file", "region"]], on="file")
def summ(g):
    p = (g.score > threshold).astype(int)
    return pd.Series(dict(n=len(g), auc=roc_auc_score(g.label, g.score) if g.label.nunique() > 1 else np.nan,
                          recall=recall_score(g.label, p, zero_division=0),
                          specificity=((p == 0) & (g.label == 0)).sum() / max((g.label == 0).sum(), 1)))
by_region = pred.groupby("region").apply(summ).sort_values("n", ascending=False)
by_region.to_csv(f"{OUT}/test_by_region.csv"); by_region

## 5. Download the results

In [ ]:
for f in glob.glob(f"{OUT}/epoch_*.weights.h5"):
    os.remove(f)   # keep only the selected weights to stay small
shutil.make_archive("/content/tars_run_outputs", "zip", OUT)
from google.colab import files
files.download("/content/tars_run_outputs.zip")